# Variables

Variables in Python are names bound to objects, not boxes that store raw values. That single idea explains a large amount of real-world confusion around mutation, aliasing, memory behavior, and bugs in shared state.


## 1. Problem First

Why does this matter in real systems?

- A pipeline step mutates a list that another step still references.
- A cache entry changes unexpectedly because two names point at the same object.
- A function appears to "copy" data but actually shares memory with the caller.


In [ ]:
payload = {"status": "new"}
job = payload
job["status"] = "processed"

print(payload)


## 2. Minimal Concept

Core syntax:

- `name = value` binds a name to an object.
- Reassignment changes the binding, not the existing object.
- Multiple names can point to the same object.


## 3. Mental Model

How Python actually behaves

Think of a variable as a label attached to an object in memory. Assignment moves labels around. Mutation changes the object behind the label. If two labels reference the same mutable object, both names observe the same change.


In [ ]:
items = [1, 2, 3]
alias = items
copy_like = items[:]

print(id(items), id(alias), id(copy_like))
alias.append(4)
print(items)
print(copy_like)


## 4. Internal Mechanics

Names live in namespaces implemented as mappings. Bytecode uses operations like `STORE_NAME`, `LOAD_NAME`, `STORE_FAST`, and `LOAD_FAST` to bind and fetch references.


In [ ]:
import dis

def rebind_example():
    x = [1, 2]
    y = x
    x = [9, 9]
    return x, y

dis.dis(rebind_example)
print(rebind_example())


## 5. Edge Cases

Where it breaks:

- Mutable defaults persist across calls.
- Rebinding inside a function does not affect the caller's name.
- In-place mutation does affect shared objects.
- Small integers and interned strings may look shared due to implementation optimizations.


In [ ]:
def append_tag(tag, tags=[]):
    tags.append(tag)
    return tags

print(append_tag("api"))
print(append_tag("db"))


## 6. Performance Thinking

- Assignment is effectively O(1) because it binds a reference.
- Shallow copying a list is O(n) because references must be copied.
- Deep copying nested structures can become expensive in both time and memory.
- Avoid unnecessary copies in hot paths, but never trade correctness for imagined savings.


## 7. Real Use Case

A data pipeline enriches records before batching. If each stage mutates the same record object, downstream systems may observe partially transformed state.


In [ ]:
record = {"id": 101, "tags": []}

def enrich(r):
    updated = dict(r)
    updated["tags"] = r["tags"] + ["validated"]
    return updated

new_record = enrich(record)
print(record)
print(new_record)


## 8. Anti-Pattern

What beginners do wrong:

- Say "Python passes by reference" without precision.
- Mutate inputs casually inside helper functions.
- Copy containers shallowly and assume nested data is isolated.


In [ ]:
# Hidden side effect
def normalize(config):
    config["timeout"] = int(config.get("timeout", 30))
    return config

original = {"timeout": "15"}
normalized = normalize(original)
print(original is normalized)
print(original)


## 9. Interview Signals

Questions FAANG asks:

- Are variables boxes or labels in Python?
- What is the difference between rebinding and mutation?
- Why does `a = b = []` often create bugs?
- What does `id()` prove and what does it not prove?


## 10. Exercise (Non-trivial)

You are debugging a batch processor where retry metadata leaks from one job into another. Find the shared-state bug, redesign the function to avoid aliasing, and explain the memory behavior that caused it.


In [ ]:
def build_job(name, metadata={}):
    metadata["created_by"] = "scheduler"
    return {"name": name, "metadata": metadata}

# Task:
# 1. Explain why jobs share metadata.
# 2. Fix the function.
# 3. Show how you would test it.
